**Script for applying the final selected (HMM) model to subject-level data, decoding network states and generating ML features.**

For INPUT, this script will scan for a subdirectoy called 'FINAL_MODEL' in the base directory for the current dataset within the main model outputs directory:
- Current dataset definition --> CONFIG fields 'ML_training.dataset_selector' (if automatic dataset targeting) AND/OR 'ML_training.dataset_path' (if manual dataset targeting)
- Current main model outputs directory --> CONFIG fields 'HMM_training.HMM_model_directory'

Provided the previous scripts have been run correctly, the assumption is that there will be a 'FINAL_MODEL' subdirectory which contains a single model file, called 'final_model.job' (and associated files, e.g. PCA settings if these were used), the provenance and other details of which are provided by the various metadata sidecar files (e.g. 'feature columns' and 'final model selection' JSON files) in this same subfolder.

The script then applies the model to the original data, assigns state labels, calculates a wide variety of related metrics/features (including both "time-resolved" and "summary-level" features), then exports the result as an ML-ready output file called '_flattened_features_ALL.csv' in the target output directory ([HMM_decoding][output_directory_name] YAML field).

[Runtime: Trivial]

----------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, subprocess
import time
from pathlib import Path
import json
import pandas as pd
import numpy as np
import joblib

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']


### PROCESSING PARAMETERS:

hmm_decoding_config = config.get("HMM_decoding", None)
if hmm_decoding_config is None:
    raise RuntimeError("[INIT ERROR] Missing config block: HMM_decoding.")
    hmm_decoding_config = {}
EXPORT_LEVEL = str(hmm_decoding_config.get("export_level", "session")).strip().lower()
if EXPORT_LEVEL not in {"session", "subject"}:
    raise ValueError(f"[INIT ERROR] HMM_decoding.export_level must be 'session' or 'subject' (got: {EXPORT_LEVEL})")
SAVE_TIME_RESOLVED_TABLE = bool(hmm_decoding_config.get("save_time_resolved", True))
SAVE_POSTERIOR_PROBABILITIES = bool(hmm_decoding_config.get("save_posterior_probabilities", True))
POSTERIOR_PROBABILITY_FORMAT = str(
    hmm_decoding_config.get("posterior_probability_format", "wide")).strip().lower()
if POSTERIOR_PROBABILITY_FORMAT not in {"wide", "json"}:
    raise ValueError("[INIT ERROR] HMM_decoding.posterior_probability_format must be 'wide' or 'json' "
        f"(got: {POSTERIOR_PROBABILITY_FORMAT})")
OVERWRITE_OUTPUT_FILES = bool(hmm_decoding_config.get("overwrite", False))

ENTROPY_MODE = config['HMM_feature_engineering']['entropy_mode']
ENTROPY_TOP_K = config['HMM_feature_engineering']['entropy_num_top_windows']
ENTROPY_THRESHOLD = config['HMM_feature_engineering']['entropy_threshold_percentile']
if not isinstance(ENTROPY_THRESHOLD, int) or not (50 <= ENTROPY_THRESHOLD <= 100):
    raise ValueError(f"[INIT ERROR] ENTROPY_THRESHOLD must be an integer between 50 and 100 (got: {ENTROPY_THRESHOLD})")

ADD_EXPLORATORY_FEATURES = config['HMM_feature_engineering']['include_auxiliary_features']


# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = BASE_DIRECTORY / 'fMRI_manifest.csv'


### INPUTS:

# Grab dataset-selection config variables:
DATASET_SELECTOR    = str(config["ML_training"]["dataset_selector"]).strip().lower()
DATASET_MANUAL_PATH = config["ML_training"].get("dataset_path", None)

# Original "dataset pointer" is in this directory, and it also contains the raw data ('X_train', 'X_all', etc.):
DATA_DIR = Path(BASE_DIRECTORY) / config['ML_prep']['training_data_dir']

# Actual models live here (under a directory w/ the same name as governs the pointer logic):
MODELS_DIR = Path(BASE_DIRECTORY) / config['HMM_training']['HMM_model_directory']

# [[[See next cell for actual input routing]]]


### OUTPUTS:

DECODING_OUTPUT_DIRECTORY_NAME = str(
    hmm_decoding_config.get("output_directory_name", "HMM_decoding")).strip()
if not DECODING_OUTPUT_DIRECTORY_NAME:
    raise ValueError("[INIT ERROR] HMM_decoding.output_directory_name is empty.")


# __________________________________________________________________________________________________________
### INITIALIZATION:

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)


# Print summary of active parameter settings:
print("\n[INIT] Active decoding & feature-extraction parameters (via CONFIG):")
print(f"  export_level                    = '{EXPORT_LEVEL}'")
print(f"  save_time_resolved              = {SAVE_TIME_RESOLVED_TABLE}")
print(f"  save_posterior_probabilities    = {SAVE_POSTERIOR_PROBABILITIES}")
print(f"  posterior_probability_format    = '{POSTERIOR_PROBABILITY_FORMAT}'")
print(f"  output_directory_name           = '{DECODING_OUTPUT_DIRECTORY_NAME}'")
print(f"  overwrite                       = {OVERWRITE_OUTPUT_FILES}")

Scan for FINAL_MODEL:

In [ ]:
# __________________________________________________________________________________________________________
### RESOLVE DATASET + LOAD FINAL_MODEL BUNDLE + LOCATE X_all
#
# Creates:
#   - DATASET_NAME, DATASET_DIR
#   - DATASET_MODELS_DIR
#   - FINAL_DIR
#   - final_model, final_pca (or None)
#   - feature_columns_expected (or None)
#   - x_all_path, provenance_path
#   - final_selection_payload (dict, best-effort)
# __________________________________________________________________________________________________________

def handle_error(message: str):
    if HARD_STOP:
        raise RuntimeError(message)
    else:
        print(f"[WARN] {message}")

# Resolve 'DATASET_NAME' and 'DATASET_DIR':
DATASET_NAME = None
DATASET_DIR = None

if DATASET_SELECTOR == "latest":
    pointer_path = Path(DATA_DIR) / "LATEST_DATASET.json"
    if not pointer_path.exists():
        raise FileNotFoundError(
            f"[INIT ERROR] dataset_selector='latest' but pointer file not found:\n  {pointer_path}")
    with open(pointer_path, "r") as f:
        pointer = json.load(f)
    pointer_dataset_dir = Path(pointer.get("dataset_dir", "")).expanduser()
    if pointer_dataset_dir is None or str(pointer_dataset_dir).strip() == "":
        raise RuntimeError(
            f"[INIT ERROR] Pointer file exists but missing/empty 'dataset_dir':\n  {pointer_path}")
    DATASET_NAME = pointer_dataset_dir.name
    DATASET_DIR = Path(DATA_DIR) / DATASET_NAME  # enforce rooting under DATA_DIR
elif DATASET_SELECTOR == "manual":
    if DATASET_MANUAL_PATH is None or str(DATASET_MANUAL_PATH).strip() == "":
        raise ValueError(
            "[INIT ERROR] dataset_selector='manual' but ML_training.dataset_path is empty.")
    manual_dir = Path(str(DATASET_MANUAL_PATH)).expanduser()
    DATASET_NAME = manual_dir.name
    DATASET_DIR = Path(DATA_DIR) / DATASET_NAME  # enforce rooting under DATA_DIR
else:
    raise ValueError(
        f"[INIT ERROR] ML_training.dataset_selector must be 'latest' or 'manual' "
        f"(got: {DATASET_SELECTOR})")
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"[INIT ERROR] DATASET_DIR not found: {DATASET_DIR}")

# Resolve dataset-specific models directory + 'FINAL_MODEL' directory:
DATASET_MODELS_DIR = Path(MODELS_DIR) / DATASET_NAME
if not DATASET_MODELS_DIR.exists():
    raise FileNotFoundError(
        "[INIT ERROR] Dataset models directory not found.\n"
        f"  Expected: {DATASET_MODELS_DIR}")

FINAL_DIR = DATASET_MODELS_DIR / "FINAL_MODEL"
if not FINAL_DIR.exists():
    raise FileNotFoundError(
        "[INIT ERROR] FINAL_MODEL directory not found.\n"
        f"  Expected: {FINAL_DIR}\n"
        "Run the cross-model evaluation script and export a final-final model first.")

# Set expected 'FINAL_MODEL' artifacts:
final_model_path = FINAL_DIR / "final_model.joblib"
final_pca_path = FINAL_DIR / "final_pca.joblib"
feature_cols_path = FINAL_DIR / "feature_columns.json"
final_sel_path = FINAL_DIR / "final_model_selection.json"

if not final_model_path.exists():
    raise FileNotFoundError(f"[INIT ERROR] Missing required file: {final_model_path}")

### Load model (+ optional PCA) & metadata:
try:
    final_model = joblib.load(final_model_path)
except Exception as exc:
    raise RuntimeError(f"[INIT ERROR] Failed to load final_model.joblib: {exc}")

final_pca = None
if final_pca_path.exists():
    try:
        final_pca = joblib.load(final_pca_path)
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to load final_pca.joblib; proceeding without PCA. Error: {exc}")
        final_pca = None

feature_columns_expected = None
if feature_cols_path.exists():
    try:
        with open(feature_cols_path, "r") as f:
            tmp = json.load(f)
        if isinstance(tmp, dict) and isinstance(tmp.get("feature_columns", None), list) and tmp["feature_columns"]:
            feature_columns_expected = list(tmp["feature_columns"])
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to read feature_columns.json; will infer from X_all/provenance. Error: {exc}")

final_selection_payload = None
if final_sel_path.exists():
    try:
        with open(final_sel_path, "r") as f:
            final_selection_payload = json.load(f)
    except Exception as exc:
        handle_error(f"[INIT WARN] Failed to read final_model_selection.json. Error: {exc}")

# Resolve frozen data inputs ('X_all' + provenance):
x_all_path = DATASET_DIR / "X_all.csv"
provenance_path = DATASET_DIR / "provenance.json"

missing = []
if not x_all_path.exists():
    missing.append(str(x_all_path))
if not provenance_path.exists():
    # Note: provenance is useful, but not strictly required if 'feature_columns.json' exists
    handle_error(f"[INIT WARN] provenance.json not found (will proceed): {provenance_path}")

if missing:
    raise FileNotFoundError("[INIT ERROR] Missing required frozen-data file(s):\n  - " + "\n  - ".join(missing))


# ----------------------------------------------------------------
# Print reports:
print("\n[INIT] Dataset + FINAL_MODEL bundle resolved:")
print(f"   DATASET_SELECTOR     = '{DATASET_SELECTOR}'")
print(f"   DATASET_NAME         = {DATASET_NAME}\n")
print(f"   DATASET_DIR          = {DATASET_DIR}")
print(f"     X_all.csv          = {x_all_path}")
print(f"     provenance.json    = {provenance_path} {'(missing)' if not provenance_path.exists() else ''}\n")
print(f"   DATASET_MODELS_DIR   = {DATASET_MODELS_DIR}")
print(f"   FINAL_DIR            = {FINAL_DIR}")
print(f"     final_model.joblib = {final_model_path}")
print(f"     final_pca.joblib   = {final_pca_path} {'(not present)' if not final_pca_path.exists() else ''}")
print(f"     feature_columns    = {feature_cols_path} {'(not present)' if not feature_cols_path.exists() else ''}")
print(f"     selection.json     = {final_sel_path} {'(not present)' if not final_sel_path.exists() else ''}")
if final_selection_payload and isinstance(final_selection_payload, dict):
    print("\n[INIT] Final selection summary:")
    print(f"   selected_protocol = {final_selection_payload.get('selected_protocol', None)}")
    print(f"   selected_K        = {final_selection_payload.get('selected_K', None)}")
    print(f"   PCA enabled       = {final_selection_payload.get('protocol_config_effective', {}).get('PCA', {}).get('enabled', None)}")

In [ ]:
# __________________________________________________________________________________________________________
### LOAD X_all + VALIDATE SCHEMA + REPORT DATASET STRUCTURE (subjects/sessions/samples)

# Load 'X_all':
x_all_df = pd.read_csv(x_all_path)
print(f"\n[DATA] Loaded X_all.csv: shape = {x_all_df.shape}")

# Check for required ID columns:
required_id_columns = ["subject_ID", "session_ID", "time_index"]
missing_id_columns = [column_name for column_name in required_id_columns if column_name not in x_all_df.columns]
if missing_id_columns:
    raise RuntimeError(f"[ERROR] 'X_all.csv' is missing required ID columns: {missing_id_columns}")

# Resolve feature columns (must be explicit & ordered):
if feature_columns_expected is None:
    # Conservative fallback: infer all non-ID columns
    #      (Ideally, 'feature_columns.json' should always exist in 'FINAL_MODEL'...)
    inferred_feature_columns = [column_name for column_name in x_all_df.columns if column_name not in required_id_columns]
    handle_error(
        "[WARNING] feature_columns_expected is None; inferring feature columns as all non-ID columns. "
        "     --> This is allowed, but FINAL_MODEL should ideally include feature_columns.json.")
    feature_columns_expected = inferred_feature_columns

missing_feature_columns = [column_name for column_name in feature_columns_expected if column_name not in x_all_df.columns]
if missing_feature_columns:
    raise RuntimeError(
        f"[ERROR] 'X_all.csv' is missing {len(missing_feature_columns)} expected feature columns. "
        f"     --> Examples: {missing_feature_columns[:10]}")

extra_non_id_columns = [
    column_name for column_name in x_all_df.columns
    if column_name not in required_id_columns and column_name not in feature_columns_expected]
if extra_non_id_columns:
    print(f"[DATA INFO] 'X_all.csv' contains {len(extra_non_id_columns)} extra non-ID columns not in feature_columns.json; they will be ignored.")
    print(f"   --> Examples: {extra_non_id_columns[:10]}")

# Enforce feature column order explicitly:
feature_df = x_all_df.loc[:, feature_columns_expected].copy()
print(f"[DATA] Feature matrix (pre-PCA) shape = {feature_df.shape}")

# Dataset structure summary:
unique_subject_ids = x_all_df["subject_ID"].dropna().unique()
number_of_subjects = len(unique_subject_ids)

# Sessions per subject:
sessions_per_subject_series = (
    x_all_df
    .groupby("subject_ID", dropna=False)["session_ID"]
    .nunique(dropna=False))

sessions_per_subject_min = int(sessions_per_subject_series.min())
sessions_per_subject_max = int(sessions_per_subject_series.max())

# Session value counts overall:
session_id_value_counts = (
    x_all_df.groupby("session_ID", dropna=False)
            .apply(lambda group_df: group_df[["subject_ID", "session_ID"]].drop_duplicates().shape[0])
            .rename("number_of_sequences"))

# Samples per [subject_ID x session_ID]
samples_per_sequence = (
    x_all_df
    .groupby(["subject_ID", "session_ID"], dropna=False)
    .size()
    .rename("number_of_windows")
    .reset_index())

samples_per_sequence_min = int(samples_per_sequence["number_of_windows"].min())
samples_per_sequence_max = int(samples_per_sequence["number_of_windows"].max())
samples_per_sequence_unique_values = samples_per_sequence["number_of_windows"].nunique()

print("\n[DATA] Dataset structure summary:")
print(f"  Number of unique subjects                 = {number_of_subjects}")
print(f"  Number of sessions per subject (min/max)  = {sessions_per_subject_min} / {sessions_per_subject_max}")

print("\n[DATA] session_ID value counts (all rows):")
print(session_id_value_counts)

print("\n[DATA] Number of windows per (subject_ID, session_ID):")
print(f"  min / max                                 = {samples_per_sequence_min} / {samples_per_sequence_max}")
print(f"  # unique window-count values              = {samples_per_sequence_unique_values}")

if samples_per_sequence_unique_values == 1:
    print("  Uniformity check                          = PASS (all sequences have the same number of windows)")
else:
    # This *might* be acceptable (e.g., dropouts), but give an explicit warning message:
    message = (
        "[DATA WARN] Non-uniform number of windows per (subject_ID, session_ID). "
        "    - This can be valid (missing data) but should be expected/intentional. "
        "    - Inspect samples_per_sequence to identify the outliers.")
    handle_error(message)
    print("\n[DATA] Example sequences with non-modal window counts:")
    modal_count = int(samples_per_sequence["number_of_windows"].mode().iloc[0])
    example_outliers = samples_per_sequence.loc[samples_per_sequence["number_of_windows"] != modal_count].head(10)
    print(example_outliers)

# Also summarize windows per 'session_ID' (across all subjects):
windows_per_session_id = (
    x_all_df.groupby("session_ID", dropna=False).size().rename("number_of_rows").sort_values(ascending=False))
print("\n[DATA] Total number of rows per session_ID (across all subjects):")
print(windows_per_session_id)

# Prepare the numeric matrix for downstream decoding (next cell will apply PCA + decode):
feature_matrix = feature_df.to_numpy(dtype=np.float64, copy=False)
if not np.isfinite(feature_matrix).all():
    non_finite_count = np.isnan(feature_matrix).sum() + np.isinf(feature_matrix).sum()
    raise RuntimeError(f"[DATA ERROR] feature_matrix contains non-finite values (NaN/Inf). Count = {non_finite_count}")

print(f"\n[DATA] feature_matrix prepared: shape = {feature_matrix.shape} | dtype = {feature_matrix.dtype}")

In [ ]:
# __________________________________________________________________________________________________________
### APPLY PCA (IF PRESENT) + DECODE STATES (+ POSTERIOR PROBABILITIES)

# Apply PCA transform (if present):
if final_pca is not None:
    try:
        feature_matrix_transformed = final_pca.transform(feature_matrix)
        print(f"[PCA] Applied PCA transform: {feature_matrix.shape} -> {feature_matrix_transformed.shape}")
    except Exception as exception:
        raise RuntimeError(f"[PCA ERROR] Failed to apply final_pca.transform(): {exception}")
else:
    feature_matrix_transformed = feature_matrix
    print(f"[PCA] No PCA found; using raw feature_matrix: shape = {feature_matrix_transformed.shape}")

# Verify model-decoding capabilities:
model_supports_predict = callable(getattr(final_model, "predict", None))
model_supports_predict_proba = callable(getattr(final_model, "predict_proba", None))

if not model_supports_predict:
    raise RuntimeError("[MODEL ERROR] final_model does not implement predict(); cannot decode state labels.")

should_save_posteriors = bool(SAVE_POSTERIOR_PROBABILITIES and model_supports_predict_proba)

print("\n[MODEL] Decoding capabilities:")
print(f"  predict() available             = {model_supports_predict}")
print(f"  predict_proba() available       = {model_supports_predict_proba}")
print(f"  save_posterior_probabilities    = {SAVE_POSTERIOR_PROBABILITIES}")
print(f"  will_compute_posteriors         = {should_save_posteriors}")

# Infer number of states K:
number_of_states = None
for attribute_name in ["n_components", "n_states", "K"]:
    if hasattr(final_model, attribute_name):
        try:
            number_of_states = int(getattr(final_model, attribute_name))
            break
        except Exception:
            pass

if should_save_posteriors and number_of_states is None:
    # probe 'predict_proba()' functionality on a small batch to infer K parameter:
    probe_size = min(10, len(feature_matrix_transformed))
    try:
        posterior_probe = final_model.predict_proba(feature_matrix_transformed[:probe_size, :])
        if isinstance(posterior_probe, np.ndarray) and posterior_probe.ndim == 2:
            number_of_states = int(posterior_probe.shape[1])
    except Exception as exception:
        raise RuntimeError(f"[MODEL ERROR] Unable to infer number_of_states (K) from predict_proba probe: {exception}")

if number_of_states is None:
    # For hard-only decoding, we can infer after decoding -- but prefer to know now if possible:
    handle_error("[MODEL WARN] Could not infer number_of_states (K) from model attributes; will infer from decoded labels.")

print(f"\n[MODEL] Inferred number_of_states (K) = {number_of_states}")

# Build a row-aligned decoded output dataframe:
decoded_time_resolved_df = x_all_df.loc[:, ["subject_ID", "session_ID", "time_index"]].copy()
decoded_time_resolved_df["row_index_original"] = np.arange(len(decoded_time_resolved_df), dtype=int)

# Sort so we can decode sequence-by-sequence, in temporal order:
sort_columns = ["subject_ID", "session_ID", "time_index"]
decoded_sorted_df = decoded_time_resolved_df.sort_values(sort_columns).reset_index(drop=True)

sorted_row_indices = decoded_sorted_df["row_index_original"].to_numpy(dtype=int)
feature_matrix_sorted = feature_matrix_transformed[sorted_row_indices, :]

# Allocate outputs:
state_label_array = np.full(len(decoded_sorted_df), fill_value=-1, dtype=int)

posterior_probability_matrix = None
if should_save_posteriors:
    if number_of_states is None:
        raise RuntimeError("[MODEL ERROR] should_save_posteriors=True but number_of_states could not be inferred.")
    posterior_probability_matrix = np.full((len(decoded_sorted_df), number_of_states), np.nan, dtype=np.float64)

# Decode per [subject_ID x session_ID]:
sequence_group_indices = decoded_sorted_df.groupby(["subject_ID", "session_ID"], dropna=False, sort=False).indices
number_of_sequences = len(sequence_group_indices)

print(f"\n[DECODE] Decoding {number_of_sequences} sequences (grouped by subject_ID x session_ID) ...")

for sequence_index, (sequence_key, index_list) in enumerate(sequence_group_indices.items(), start=1):
    row_index_array = np.array(index_list, dtype=int)
    sequence_feature_matrix = feature_matrix_sorted[row_index_array, :]

    try:
        state_label_array[row_index_array] = final_model.predict(sequence_feature_matrix)
    except Exception as exception:
        raise RuntimeError(
            f"[DECODE ERROR] predict() failed for sequence={sequence_key} "
            f"(n_windows={len(row_index_array)}): {exception}")

    if should_save_posteriors:
        try:
            posterior_probability_matrix[row_index_array, :] = final_model.predict_proba(sequence_feature_matrix)
        except Exception as exception:
            handle_error(
                f"[DECODE WARN] predict_proba() failed for sequence={sequence_key}; "
                f"posterior probabilities will be NaN for this sequence. Error: {exception}")

    if sequence_index <= 3 or sequence_index == number_of_sequences:
        print(f"  decoded {sequence_index}/{number_of_sequences}: key={sequence_key}, n_windows={len(row_index_array)}")

decoded_sorted_df["state_label"] = state_label_array

# If we didn't already know K, infer it from decoded labels now:
if number_of_states is None:
    if (state_label_array >= 0).any():
        number_of_states = int(state_label_array.max() + 1)
        print(f"[MODEL] Inferred number_of_states (K) from decoded labels: {number_of_states}")
    else:
        raise RuntimeError("[DECODE ERROR] All decoded state labels are negative; decoding failed unexpectedly.")

# Attach posterior outputs (optional) + confidence metrics:
if should_save_posteriors:
    if POSTERIOR_PROBABILITY_FORMAT == "wide":
        for state_index in range(number_of_states):
            column_name = f"posterior_probability_state_{state_index:02d}"
            decoded_sorted_df[column_name] = posterior_probability_matrix[:, state_index]
    elif POSTERIOR_PROBABILITY_FORMAT == "json":
        decoded_sorted_df["posterior_probability_json"] = [
            json.dumps(row.tolist()) if np.isfinite(row).all() else None
            for row in posterior_probability_matrix]
    else:
        raise RuntimeError(f"[DECODE ERROR] Unexpected POSTERIOR_PROBABILITY_FORMAT: {POSTERIOR_PROBABILITY_FORMAT}")

    # Confidence summaries:
    epsilon = 1e-12
    posterior_probability_matrix_clipped = np.clip(posterior_probability_matrix, epsilon, 1.0)
    decoded_sorted_df["posterior_probability_max"] = np.nanmax(posterior_probability_matrix_clipped, axis=1)
    decoded_sorted_df["posterior_probability_entropy"] = -np.nansum(
        posterior_probability_matrix_clipped * np.log(posterior_probability_matrix_clipped),
        axis=1)

# Restore original row order (aligned to 'X_all'):
decoded_time_resolved_df = (
    decoded_sorted_df
    .sort_values("row_index_original")
    .drop(columns=["row_index_original"])
    .reset_index(drop=True))

print("\n[DECODE] Time-resolved decoding complete.")
print(decoded_time_resolved_df.head())

In [ ]:
# __________________________________________________________________________________________________________
### HMM SUMMARY FEATURE EXTRACTION (SESSION-LEVEL):

def compute_run_lengths(state_label_array: np.ndarray) -> np.ndarray:
    """Return run lengths of consecutive identical state labels."""
    if len(state_label_array) == 0:
        return np.array([], dtype=int)
    run_lengths = []
    current_state = state_label_array[0]
    current_length = 1
    for next_state in state_label_array[1:]:
        if next_state == current_state:
            current_length += 1
        else:
            run_lengths.append(current_length)
            current_state = next_state
            current_length = 1
    run_lengths.append(current_length)
    return np.asarray(run_lengths, dtype=int)


def compute_transition_probability_matrix(state_label_array: np.ndarray, number_of_states: int) -> np.ndarray:
    """Compute row-normalized transition probability matrix."""
    transition_counts = np.zeros((number_of_states, number_of_states), dtype=float)
    if len(state_label_array) < 2:
        return transition_counts
    for from_state, to_state in zip(state_label_array[:-1], state_label_array[1:]):
        if from_state >= 0 and to_state >= 0:
            transition_counts[int(from_state), int(to_state)] += 1.0
    row_sums = transition_counts.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        transition_probabilities = np.divide(
            transition_counts,
            row_sums,
            out=np.zeros_like(transition_counts),
            where=row_sums > 0)
    return transition_probabilities


summary_rows = []

grouped_sequences = (decoded_time_resolved_df.sort_values(["subject_ID", "session_ID", "time_index"]).groupby(["subject_ID", "session_ID"], dropna=False))

# Guardrails / numerical stability:
epsilon = 1e-12

# Entropy feature engineering configuration:
if not isinstance(ENTROPY_MODE, str) or ENTROPY_MODE.strip().lower() not in {"top_windows", "percentile_threshold"}:
    raise ValueError(
        f"[INIT ERROR] ENTROPY_MODE must be 'top_windows' or 'percentile_threshold' (got: {ENTROPY_MODE})")
ENTROPY_MODE = ENTROPY_MODE.strip().lower()

if ENTROPY_MODE == "top_windows":
    if not isinstance(ENTROPY_TOP_K, int) or ENTROPY_TOP_K < 1:
        raise ValueError(
            f"[INIT ERROR] ENTROPY_TOP_K must be a positive integer when ENTROPY_MODE='top_windows' (got: {ENTROPY_TOP_K})")
    entropy_high_ambiguity_column_name = f"mean_entropy_top_{ENTROPY_TOP_K:02d}_windows"
else:
    if not isinstance(ENTROPY_THRESHOLD, int) or not (50 <= ENTROPY_THRESHOLD <= 100):
        raise ValueError(
            f"[INIT ERROR] ENTROPY_THRESHOLD must be an integer between 50 and 100 when "
            f"ENTROPY_MODE='percentile_threshold' (got: {ENTROPY_THRESHOLD})")
    entropy_high_ambiguity_column_name = f"fraction_high_entropy_windows_p{ENTROPY_THRESHOLD:02d}"

log_entropy_transition_ratio_column_name = "log_entropy_transition_ratio"

for (subject_id, session_id), sequence_df in grouped_sequences:

    state_labels = sequence_df["state_label"].to_numpy(dtype=int)

    summary_row = {
        "subject_ID": subject_id,
        "session_ID": session_id,
        "number_of_windows": len(sequence_df)}

    # Calculate "hard" state occupancies:
    state_counts = np.bincount(state_labels, minlength=number_of_states).astype(float)
    state_occupancy = state_counts / state_counts.sum()
    for state_index in range(number_of_states):
        summary_row[f"occupancy-Hard_state-{state_index:02d}"] = state_occupancy[state_index]

    # Calculate dwell- & run-length statistics:
    run_lengths = compute_run_lengths(state_labels)
    summary_row["dwell_mean"] = float(np.mean(run_lengths))
    summary_row["dwell_median"] = float(np.median(run_lengths))
    summary_row["dwell_max"] = float(np.max(run_lengths))
    summary_row["number_of_state_runs"] = float(len(run_lengths))

    # Calculate transition probabilities:
    transition_probability_matrix = compute_transition_probability_matrix(state_labels, number_of_states)
    for from_state in range(number_of_states):
        for to_state in range(number_of_states):
            summary_row[f"transition_probability_state_{from_state:02d}_to_{to_state:02d}"] = transition_probability_matrix[from_state, to_state]

    # Calculate posterior confidence summaries (if available):
    if "posterior_probability_max" in sequence_df.columns:
        summary_row["mean_posterior_probability_max"] = float(
            sequence_df["posterior_probability_max"].mean())

    if "posterior_probability_entropy" in sequence_df.columns:
        posterior_entropy = sequence_df["posterior_probability_entropy"].to_numpy(dtype=float)

        # Existing summary metrics:
        summary_row["mean_posterior_probability_entropy"] = float(np.mean(posterior_entropy))
        summary_row["std_posterior_probability_entropy"] = (
            float(np.std(posterior_entropy, ddof=1)) if len(posterior_entropy) > 1 else np.nan)

        # NEW --> "High ambiguity" entropy summary (mode-dependent):
        if len(posterior_entropy) == 0:
            summary_row[entropy_high_ambiguity_column_name] = np.nan
        else:
            if ENTROPY_MODE == "top_windows":
                # Calculate the mean of the top-K entropy windows (K clipped to available windows):
                number_of_windows_available = int(len(posterior_entropy))
                number_of_top_windows = int(min(ENTROPY_TOP_K, number_of_windows_available))
                if number_of_top_windows < 1:
                    summary_row[entropy_high_ambiguity_column_name] = np.nan
                else:
                    top_entropy_values = np.sort(posterior_entropy)[-number_of_top_windows:]
                    summary_row[entropy_high_ambiguity_column_name] = float(np.mean(top_entropy_values))
            elif ENTROPY_MODE == "percentile_threshold":
                # Calculate the fraction of windows above a within-session percentile threshold:
                entropy_percentile_threshold_value = float(np.percentile(posterior_entropy, ENTROPY_THRESHOLD))
                summary_row[entropy_high_ambiguity_column_name] = float(
                    np.mean(posterior_entropy >= entropy_percentile_threshold_value))
            else:
                raise RuntimeError(f"[INIT ERROR] Unexpected ENTROPY_MODE: {ENTROPY_MODE}")

        # Entropy around transitions vs within runs:
        #       Define transition windows using changes in hard state label; for t>=2, transition occurs at time 't' if state(t) != state(t-1).
        #       We'll align entropy with the "current" time index (i.e. entropy at t)...
        if len(state_labels) >= 2:
            transition_indicator = (state_labels[1:] != state_labels[:-1])  # <-- length = 'n_windows' - 1
            entropy_aligned = posterior_entropy[1:]  # entropy at windows 2..T
            transition_entropy_values = entropy_aligned[transition_indicator]
            non_transition_entropy_values = entropy_aligned[~transition_indicator]
            summary_row["mean_entropy_transition_windows"] = (
                float(np.mean(transition_entropy_values)) if len(transition_entropy_values) > 0 else np.nan)
            summary_row["mean_entropy_non_transition_windows"] = (
                float(np.mean(non_transition_entropy_values)) if len(non_transition_entropy_values) > 0 else np.nan)
            if np.isfinite(summary_row["mean_entropy_transition_windows"]) and np.isfinite(summary_row["mean_entropy_non_transition_windows"]):
                entropy_transition_ratio_value = (
                    summary_row["mean_entropy_transition_windows"] / (summary_row["mean_entropy_non_transition_windows"] + epsilon))
                summary_row[log_entropy_transition_ratio_column_name] = float(np.log(entropy_transition_ratio_value + epsilon))
            else:
                summary_row[log_entropy_transition_ratio_column_name] = np.nan
        else:
            summary_row["mean_entropy_transition_windows"] = np.nan
            summary_row["mean_entropy_non_transition_windows"] = np.nan
            summary_row[log_entropy_transition_ratio_column_name] = np.nan
    else:
        # If entropy was not computed/saved, explicitly add all-NaN columns to maintain overall table schema stability:
        summary_row["mean_posterior_probability_entropy"] = np.nan
        summary_row["std_posterior_probability_entropy"] = np.nan
        summary_row[entropy_high_ambiguity_column_name] = np.nan
        summary_row["mean_entropy_transition_windows"] = np.nan
        summary_row["mean_entropy_non_transition_windows"] = np.nan
        summary_row[log_entropy_transition_ratio_column_name] = np.nan
    summary_rows.append(summary_row)

hmm_summary_session_df = pd.DataFrame(summary_rows)

print("\n[SUMMARY] Session-level HMM feature table created:")
print(f"  shape = {hmm_summary_session_df.shape}")

print("\nSample rows of output:")
hmm_summary_session_df.head()

--------
### Final save/export

- First cell exports two main tables, comprising the "time-resolved" and "summary-level" features, respectively (not flattened).
- Second cell combines all features into a single flattened, ML-ready table called '_flattened_features_ALL.csv'

In [ ]:
# __________________________________________________________________________________________________________
### EXPORT RESULTS (RESPECT EXPORT_LEVEL):

# We need 'DATASET_DIR' so we are doing this here, rather than in initialization section:
output_directory = BASE_DIRECTORY / DECODING_OUTPUT_DIRECTORY_NAME / DATASET_NAME
output_directory.mkdir(parents=True, exist_ok=True)

def write_dataframe_safely(dataframe: pd.DataFrame, output_path: Path):
    if output_path.exists() and not OVERWRITE_OUTPUT_FILES:
        raise RuntimeError(
            f"[WRITE ERROR] Refusing to overwrite existing file: {output_path} "
            "(set HMM_decoding.overwrite = true to allow).")
    dataframe.to_csv(output_path, index=False)

# -----------------------------------
# Write time-resolved output:
# -----------------------------------
if SAVE_TIME_RESOLVED_TABLE:
    time_resolved_output_path = output_directory / "decoded_time_resolved.csv"
    write_dataframe_safely(decoded_time_resolved_df, time_resolved_output_path)
    print(f"[WRITE] Time-resolved decoding written to: {time_resolved_output_path}")

# ---------------------------------------
# Write summary output (session-level or subject-level):
# ---------------------------------------
if EXPORT_LEVEL == "session":
    summary_output_df = hmm_summary_session_df.copy()

elif EXPORT_LEVEL == "subject":
    # Collapse across sessions per subject:
    #     (Continuous features --> mean; window counts --> sum)
    aggregation_map = {}
    for column_name in hmm_summary_session_df.columns:
        if column_name == "subject_ID":
            continue
        elif column_name == "number_of_windows":
            aggregation_map[column_name] = "sum"
        elif column_name == "session_ID":
            continue
        else:
            aggregation_map[column_name] = "mean"

    summary_output_df = (
        hmm_summary_session_df
        .groupby("subject_ID", dropna=False)
        .agg(aggregation_map)
        .reset_index())

else:
    raise RuntimeError(f"[EXPORT ERROR] Unexpected EXPORT_LEVEL: {EXPORT_LEVEL}")

summary_output_path = output_directory / "HMM_summary_features.csv"
write_dataframe_safely(summary_output_df, summary_output_path)
print(f"[WRITE] Summary HMM features written to: {summary_output_path}")

# ---------------------------------------
# Write metadata / provenance files:
# ---------------------------------------
decoding_metadata = {
    "dataset_name": DATASET_NAME,
    "export_level": EXPORT_LEVEL,
    "number_of_states": number_of_states,
    "pca_applied": final_pca is not None,
    "time_resolved_saved": SAVE_TIME_RESOLVED_TABLE,
    "posterior_probabilities_saved": SAVE_POSTERIOR_PROBABILITIES,
    "posterior_probability_format": POSTERIOR_PROBABILITY_FORMAT,
    "number_of_subjects": int(decoded_time_resolved_df["subject_ID"].nunique()),
    "number_of_sessions": int(
        decoded_time_resolved_df[["subject_ID", "session_ID"]].drop_duplicates().shape[0])}

metadata_output_path = output_directory / "decoding_metadata.json"
if metadata_output_path.exists() and not OVERWRITE_OUTPUT_FILES:
    raise RuntimeError(f"[WRITE ERROR] Refusing to overwrite existing file: {metadata_output_path}")

with open(metadata_output_path, "w") as f:
    json.dump(decoding_metadata, f, indent=2)

print(f"[WRITE] Decoding metadata written to: {metadata_output_path}")

In [ ]:
# __________________________________________________________________________________________________________
### FINAL EXPORT: FLATTEN TIME-RESOLVED + MERGE WITH SUMMARY FEATURES (RESPECTS 'EXPORT_LEVEL'):

flattened_output_path = output_directory / "_flattened_features_ALL.csv"

# ---------------------------
# Validate 'decoded_time_resolved_df' inputs:
# ---------------------------
required_time_columns = {"subject_ID", "session_ID", "time_index", "state_label"}
missing_time_columns = required_time_columns - set(decoded_time_resolved_df.columns)
if missing_time_columns:
    raise RuntimeError(
        f"[FLATTEN ERROR] decoded_time_resolved_df missing required columns: {sorted(missing_time_columns)}")

time_resolved_df = decoded_time_resolved_df.copy()
time_resolved_df["time_index"] = time_resolved_df["time_index"].astype(int)

# Add optional time-resolved "exploratory feature" column ('posterior_probability_max'):
include_time_resolved_assigned_state_probability = bool(ADD_EXPLORATORY_FEATURES)
if include_time_resolved_assigned_state_probability and "posterior_probability_max" not in time_resolved_df.columns:
    raise RuntimeError(
        "[FLATTEN ERROR] ADD_EXPLORATORY_FEATURES=True but 'posterior_probability_max' is missing from decoded_time_resolved_df.")

# -------------------------------------------
# Build time-resolved wide table ('EXPORT_LEVEL'-dependent):
# -------------------------------------------

if EXPORT_LEVEL == "session":
    # One row per [subject_ID x session_ID], columns: 't####_state' (+ optional 't####_postProbAssigned'):
    state_wide_df = (
        time_resolved_df
        .pivot_table(
            index=["subject_ID", "session_ID"],
            columns="time_index",
            values="state_label",
            aggfunc="first"))
    state_wide_df.columns = [f"t{int(time_value):04d}_state" for time_value in state_wide_df.columns]
    state_wide_df = state_wide_df.reset_index()

    if include_time_resolved_assigned_state_probability:
        postprob_wide_df = (
            time_resolved_df
            .pivot_table(
                index=["subject_ID", "session_ID"],
                columns="time_index",
                values="posterior_probability_max",
                aggfunc="first"))
        postprob_wide_df.columns = [f"t{int(time_value):04d}_postProbAssigned" for time_value in postprob_wide_df.columns]
        postprob_wide_df = postprob_wide_df.reset_index()

        time_wide_df = state_wide_df.merge(
            postprob_wide_df,
            on=["subject_ID", "session_ID"],
            how="left",
            validate="one_to_one")
    else:
        time_wide_df = state_wide_df

elif EXPORT_LEVEL == "subject":
    # One row per subject_ID, w/ session-prefixed time columns:
    #     - '<session_ID>_t####_state' (+ optional '<session_ID>_t####_postProbAssigned')
    state_wide_df = (
        time_resolved_df
        .pivot_table(
            index=["subject_ID", "session_ID"],
            columns="time_index",
            values="state_label",
            aggfunc="first"))
    state_wide_df.columns = [f"t{int(time_value):04d}_state" for time_value in state_wide_df.columns]
    state_wide_df = state_wide_df.reset_index()

    # Prefix time columns w/ <session_ID>, then drop 'session_ID' and collapse to one row per subject:
    #        (This assumes each subject has <=1 row per session_ID; which is true by construction after pivot...)
    time_state_columns = [c for c in state_wide_df.columns if c.startswith("t") and c.endswith("_state")]
    for column_name in time_state_columns:
        state_wide_df[column_name] = state_wide_df[column_name]  # no-op, keeps dtype stable

    # Build a renamed copy with session-prefixed column names:
    state_prefixed_rows = []
    for _, row in state_wide_df.iterrows():
        subject_id_value = row["subject_ID"]
        session_id_value = str(row["session_ID"])
        renamed_row = {"subject_ID": subject_id_value}
        for column_name in time_state_columns:
            renamed_row[f"{session_id_value}_{column_name}"] = row[column_name]
        state_prefixed_rows.append(renamed_row)
    state_prefixed_df = pd.DataFrame(state_prefixed_rows)

    # If multiple sessions exist, we need to merge them horizontally per subject_ID:
    #        (Groupby+first is safe because each [subject x session] produces unique prefixed columns)
    time_wide_df = (
        state_prefixed_df
        .groupby("subject_ID", dropna=False)
        .first()
        .reset_index())

    # Optional: session-prefixed posterior probability of assigned state:
    if include_time_resolved_assigned_state_probability:
        postprob_wide_df = (
            time_resolved_df
            .pivot_table(
                index=["subject_ID", "session_ID"],
                columns="time_index",
                values="posterior_probability_max",
                aggfunc="first"))
        postprob_wide_df.columns = [f"t{int(time_value):04d}_postProbAssigned" for time_value in postprob_wide_df.columns]
        postprob_wide_df = postprob_wide_df.reset_index()

        postprob_columns = [c for c in postprob_wide_df.columns if c.startswith("t") and c.endswith("_postProbAssigned")]

        postprob_prefixed_rows = []
        for _, row in postprob_wide_df.iterrows():
            subject_id_value = row["subject_ID"]
            session_id_value = str(row["session_ID"])
            renamed_row = {"subject_ID": subject_id_value}
            for column_name in postprob_columns:
                renamed_row[f"{session_id_value}_{column_name}"] = row[column_name]
            postprob_prefixed_rows.append(renamed_row)
        postprob_prefixed_df = pd.DataFrame(postprob_prefixed_rows)

        postprob_prefixed_df = (
            postprob_prefixed_df
            .groupby("subject_ID", dropna=False)
            .first()
            .reset_index())

        time_wide_df = time_wide_df.merge(
            postprob_prefixed_df,
            on="subject_ID",
            how="left",
            validate="one_to_one")

else:
    raise RuntimeError(f"[FLATTEN ERROR] Unexpected EXPORT_LEVEL: {EXPORT_LEVEL}")

# -----------------------------
# Select summary-level columns (agnostic + toggle-aware):
#     IMPORTANT: use 'summary_output_df' (this already respects the current 'EXPORT_LEVEL' config parameter setting)
# -----------------------------

if "summary_output_df" not in globals():
    raise RuntimeError(
        "[FLATTEN ERROR] summary_output_df not found. Ensure final cell #1 has run and created summary_output_df.")

summary_df = summary_output_df.copy()

# Validate required IDs depending on 'EXPORT_LEVEL' setting:
if EXPORT_LEVEL == "session":
    required_summary_id_columns = {"subject_ID", "session_ID"}
else:
    required_summary_id_columns = {"subject_ID"}

missing_summary_id_columns = required_summary_id_columns - set(summary_df.columns)
if missing_summary_id_columns:
    raise RuntimeError(
        f"[FLATTEN ERROR] summary_output_df missing required ID columns: {sorted(missing_summary_id_columns)}")

# Detect "high-ambiguity entropy" column(s) agnostically:
high_ambiguity_entropy_columns = [
    column_name for column_name in summary_df.columns
    if column_name.startswith("fraction_high_entropy_windows_") or column_name.startswith("mean_entropy_top_")]

core_summary_column_candidates = [
    "mean_entropy_transition_windows",
    "mean_entropy_non_transition_windows",
    "entropy_transition_ratio",
    "log_entropy_transition_ratio",
    "mean_posterior_probability_entropy",
    "std_posterior_probability_entropy",
    "dwell_mean",
    "dwell_max"]

exploratory_summary_column_candidates = [
    "mean_posterior_probability_max",
    "dwell_median",
    "number_of_state_runs"]

# Set occupancy + transition blocks:
occupancy_columns = sorted([c for c in summary_df.columns if c.startswith("occupancy")])
transition_probability_columns = sorted([c for c in summary_df.columns if c.startswith("transition_probability_state_")])

# Build summary column list:
summary_columns_to_include = list(required_summary_id_columns)

summary_columns_to_include.extend(high_ambiguity_entropy_columns)

for column_name in core_summary_column_candidates:
    if column_name in summary_df.columns:
        if column_name == "log_entropy_transition_ratio" and "entropy_transition_ratio" in summary_df.columns:
            continue
        summary_columns_to_include.append(column_name)

if ADD_EXPLORATORY_FEATURES:
    for column_name in exploratory_summary_column_candidates:
        if column_name in summary_df.columns:
            summary_columns_to_include.append(column_name)

summary_columns_to_include.extend(occupancy_columns)
summary_columns_to_include.extend(transition_probability_columns)

# De-duplicate while preserving order:
seen = set()
summary_columns_to_include = [c for c in summary_columns_to_include if not (c in seen or seen.add(c))]

summary_selected_df = summary_df.loc[:, summary_columns_to_include]

# If only log ratio exists, export it under the expected name:
if "entropy_transition_ratio" not in summary_selected_df.columns and "log_entropy_transition_ratio" in summary_selected_df.columns:
    summary_selected_df = summary_selected_df.rename(columns={"log_entropy_transition_ratio": "entropy_transition_ratio"})

# --------------------------------------------------------
# Merge summary + time-resolved table, wide format:
# --------------------------------------------------------
if EXPORT_LEVEL == "session":
    flattened_features_df = summary_selected_df.merge(
        time_wide_df,
        on=["subject_ID", "session_ID"],
        how="left",
        validate="one_to_one")
else:
    flattened_features_df = summary_selected_df.merge(
        time_wide_df,
        on=["subject_ID"],
        how="left",
        validate="one_to_one")

# Sorting:
sort_columns = ["subject_ID"] + (["session_ID"] if EXPORT_LEVEL == "session" else [])
flattened_features_df = flattened_features_df.sort_values(sort_columns).reset_index(drop=True)

# -----------------------------
# 5) Reorder columns:
#     --> 'subject_ID', then 'session_ID' (if still present), then ALL time-resolved state columns, then the rest
# -----------------------------
all_columns = list(flattened_features_df.columns)

id_columns = ["subject_ID"] + (["session_ID"] if "session_ID" in flattened_features_df.columns else [])

# Set time-resolved state columns:
#     session-mode column names: 't####_state'
#     subject-mode column names: '<sessionID>_t####_state'
time_state_columns = [c for c in all_columns if c.endswith("_state") and ("_t" in c or c.startswith("t"))]

def extract_time_index_from_time_state_column_name(column_name: str) -> int:
    # This should handle both, e.g.: t0001_state' and 'baseline_t0001_state', etc.:
    try:
        if "_t" in column_name:
            time_token = column_name.split("_t")[1].split("_")[0]
        else:
            time_token = column_name.split("_")[0][1:]
        return int(time_token)
    except Exception:
        return 10**9

time_state_columns = sorted(time_state_columns, key=extract_time_index_from_time_state_column_name)

remaining_columns = [c for c in all_columns if c not in id_columns + time_state_columns]
final_column_order = id_columns + time_state_columns + remaining_columns

flattened_features_df = flattened_features_df.loc[:, final_column_order]

# ____________________________________
# Final write-out to disk:
if flattened_output_path.exists() and not OVERWRITE_OUTPUT_FILES:
    raise FileExistsError(
        f"[EXPORT ERROR] Output file already exists and overwrite=False:\n  {flattened_output_path}")

flattened_features_df.to_csv(flattened_output_path, index=False)

print("\n[EXPORT] Flattened ML-ready feature table written:")
print(f"  path  = {flattened_output_path}")
print(f"  shape = {flattened_features_df.shape}")
print("\nSample rows:")
flattened_features_df.sample(n=min(10, len(flattened_features_df)))

--------